# 01 — Data check and basic EDA

This notebook inspects one BEIR target-domain dataset before building retrieval models.

Default dataset: `beir/scifact/test`.

We will:
- load documents, queries, and relevance judgments;
- inspect a few examples;
- check basic dataset sizes;
- examine query/document length distributions;
- inspect how many relevant documents each query has.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from robust_retrieval.data.datasets import load_retrieval_dataset

In [ ]:
DATASET_NAME = "beir/scifact/test"

dataset = load_retrieval_dataset(DATASET_NAME)

print(f"Dataset:   {DATASET_NAME}")
print(f"Documents: {len(dataset.documents):,}")
print(f"Queries:   {len(dataset.queries):,}")
print(f"Qrels:     {sum(len(v) for v in dataset.qrels.values()):,}")

## Inspect a few examples

In [ ]:
list(dataset.queries.items())[:5]

In [ ]:
list(dataset.documents.items())[:3]

In [ ]:
list(dataset.qrels.items())[:5]

## Convert to DataFrames

In [ ]:
import pandas as pd

documents_df = pd.DataFrame(dataset.documents.items(), columns=["doc_id", "text"])
queries_df = pd.DataFrame(dataset.queries.items(), columns=["query_id", "query"])

qrel_rows = []
for query_id, rels in dataset.qrels.items():
    for doc_id, relevance in rels.items():
        qrel_rows.append({
            "query_id": query_id,
            "doc_id": doc_id,
            "relevance": relevance,
        })

qrels_df = pd.DataFrame(qrel_rows)

documents_df.head()

In [ ]:
queries_df.head()

In [ ]:
qrels_df.head()

## Basic length statistics

For this first pass, word count is simply whitespace-separated tokens. This is enough to understand the rough shape of the data.

In [ ]:
documents_df["word_count"] = documents_df["text"].str.split().str.len()
queries_df["word_count"] = queries_df["query"].str.split().str.len()

print("Document length summary")
display(documents_df["word_count"].describe())

print("\nQuery length summary")
display(queries_df["word_count"].describe())

In [ ]:
import matplotlib.pyplot as plt

documents_df["word_count"].hist(bins=40)
plt.title("Document length distribution")
plt.xlabel("Words per document")
plt.ylabel("Count")
plt.show()

In [ ]:
queries_df["word_count"].hist(bins=30)
plt.title("Query length distribution")
plt.xlabel("Words per query")
plt.ylabel("Count")
plt.show()

## Relevance-judgment structure

In [ ]:
relevant_per_query = (
    qrels_df[qrels_df["relevance"] > 0]
    .groupby("query_id")
    .size()
)

relevant_per_query.describe()

In [ ]:
qrels_df["relevance"].value_counts().sort_index()

## Look at one query together with its relevant documents

In [ ]:
example_query_id = next(
    query_id
    for query_id, rels in dataset.qrels.items()
    if any(rel > 0 for rel in rels.values())
)

print("Query ID:", example_query_id)
print("Query:", dataset.queries[example_query_id])
print()

for doc_id, relevance in dataset.qrels[example_query_id].items():
    if relevance > 0:
        print("Relevant document:", doc_id)
        print("Relevance:", relevance)
        print(dataset.documents[doc_id][:1200])
        print("-" * 80)

## Questions to note before modelling

After running the notebook, note:
- Are queries short or descriptive?
- Are documents mostly short passages or longer articles?
- Does each query usually have one relevant document or several?
- Are relevance labels binary or graded?
- Does the domain contain specialist terminology?
- Where might exact lexical matching help BM25?
- Where might semantic matching help a dense retriever?